# SHAP Value and interpretable ML

SHAP values are quite popular for medical purposes, like diagnostical models, since they can explain the importance of the features for one patient within a model given a sample of other patients. The sample of other patients are important, since they describe what the distribution of the different features are. A very good introduction to SHAP values can be found in [TowardsDataScience](https://www.google.com/url?sa=t&rct=j&q=&esrc=s&source=web&cd=&cad=rja&uact=8&ved=2ahUKEwjq0vSwq8f8AhW_XvEDHRM2C6EQFnoECAkQAw&url=https%3A%2F%2Ftowardsdatascience.com%2Fintroduction-to-shap-values-and-their-application-in-machine-learning-8003718e6827%23%3A~%3Atext%3DSHAP%2520is%2520a%2520mathematical%2520method%2Ceach%2520feature%2520to%2520the%2520prediction.&usg=AOvVaw0RVbH5uvfVjN2nebNqp-0Q).

A clear downside of the permutation importance, besides not working for one sample, is that it doesn't consider the feature distribution wholistically, but rather assumes each feature has an independent distribution. The permutation of features can introduce higly unrealistic fake samples used to estimate their importance.

SHAP values on the other hand, consider the effect of features working together. The SHAP-value for one feature  *i*  and one sample  *x*  is the expected value of the model difference between two samples both constructured from some of the features of  *x*  and some other from a random sample  *y* . However, one of both has inherits feature  i  from  x , and the other from  *y* . This means that if feature  *i*  for both samples is the same, the difference is *0* .

Due to the high amount of combinations possible for the expected values, they are often estimated by random sampling. Ultimately, when all SHAP-values are summed up over all features, they provide the difference between the prediction of  *x*  and the average prediction over all samples. In this sense, they describe the importance of a feature for the prediction outcome of *x* .

### 1. Download packages and data

In [ ]:
## Ordinary sklearn packages and more

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.svm import SVC

import graphviz
from sklearn import preprocessing
from matplotlib import pyplot
import pandas
import numpy
import seaborn
print("Start complete")

In [ ]:
import shap
import os

In [ ]:
'''
try: 
    from download import download
except ModuleNotFoundError:
    !pip install download
    from download import download
'''

In this application, we will use Parkinson Disease data.

In [ ]:
filepath = "/kaggle/input/parkinson-disease-detection/Parkinsson disease.csv"

data = pandas.read_csv(filepath)

In [ ]:
print(data.shape)
data.tail()

Construct the train and test data.

In [ ]:
independent_variables = [
  'MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)', 'MDVP:Jitter(%)',
  'MDVP:Jitter(Abs)', 'MDVP:RAP', 'MDVP:PPQ', 'Jitter:DDP',
  'MDVP:Shimmer', 'MDVP:Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5',
  'MDVP:APQ', 'Shimmer:DDA', 'NHR', 'HNR', 'RPDE', 'DFA',
  'spread1', 'spread2', 'D2', 'PPE'
]

health_status = data['status'].values.astype(int)

voice_statistics = data[independent_variables]

X_train, X_test, y_train, y_test = train_test_split(
    voice_statistics, health_status, test_size = 0.2, random_state=4686)

### 2. Build a model

We build a Random Forest model for this dataset.

In [ ]:
model = RandomForestClassifier(max_depth=6, n_estimators=10, random_state=4686)
model.fit(X_train, y_train)

from sklearn.metrics import classification_report
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

After training, we get a decent performance. On the test set for the participants with Parkinsson's disease the model achieved a high Recall and Accuracy rate, and the model could be a candidate to do preliminary diagnosis when verified on a bigger, random data set.

In [ ]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, y_pred))

### 3. The SHAP-values for individual samples

As introduced, the SHAP values can be calculated for individual samples. These values then explain the output of the function. We could either select the actual prediction (0 or 1) or the estimated probability (between 0 and 1). Here the latter is chosen, using model.predict_proba, the lambda-construct creates an anonymous function that extracts the probability of 1, since predict_proba also outputs the probability of 0.

The shap_values function then returns a  *n_samples×n_features*  matrix of shap values for each sample in X_test and each feature.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Create an explainer
explainer = shap.KernelExplainer(lambda x: model.predict_proba(x)[..., 1], X_test)
#explainer = shap.KernelExplainer(model.predict, X_test)

# Calculate the SHAP values
shap_values = explainer.shap_values(X_test)

In [ ]:
shap_values.shape

**The expected value from Shap is (almost) same as the mean of predict probability. **

In [ ]:
from numpy.core import shape_base
y_pred_proba = model.predict_proba(X_test)[..., 1]
print(explainer.expected_value - y_pred_proba.mean())

SHAP: Individual contributions over the average. 

In [ ]:
expected_plus_shap = explainer.expected_value + shap_values.sum(1)
expected_plus_shap - y_pred_proba

Finally, we plot the SHAP values for individual samples. 
The for-loop below iterates over indices of misclassified samples in the test set.

In [ ]:
from sys import base_exec_prefix
shap.initjs()
for error in numpy.where(y_pred != y_test)[0]:
  print(f'test sample {error:02d}.', "False Positive" if y_pred[error] == 1 else "False Negative")
  display(shap.force_plot(base_value=explainer.expected_value, shap_values=shap_values
))

**Correctly classified samples**

In [ ]:
shap.initjs()
for right in numpy.where(y_pred == y_test)[0]:
  print(f'test sample {right:02d}.', "True Positive" if y_pred[right] == 1 else "True Negative")
  display(shap.force_plot(
      explainer.expected_value,
      shap_values[right,:],
      X_test.iloc[right]
))

### 4. Global Feature importances

`summary_plot` lays out the importances of different features for all samples in one plot, each dot represents one sample and it's associated SHAP value for each sample. This means the plot contains $n_{samples} \times n_{features}$ dots. The plot shows three things:

 * the feature (vertical axis),
 * the SHAP value (horizontal axis), and
 * the relative feature value (color).

In [ ]:
shap.summary_plot(shap_values, X_test)

### 5. Coeffect of two features

Not only the summarized effects, but the dependence of two features can be showed with their shap values.

In [ ]:
Feature_of_interest = "DFA"
Other_feature = "D2"

shap.dependence_plot(Feature_of_interest, shap_values, X_test, interaction_index=Other_feature)


### 6. Some remarks

The SHAP approach provides a very useful tool to investigate the importances of different features (and samples). One downside is that it takes rather long time to calculate the SHAP values for moderate data size.  